# Logging Callbacks: stdout heartbeats vs MLflow metric history

This is the first **advanced** tutorial in this repo. It assumes you've already worked through:

- `b_tracking_quickstart.ipynb` — one model, one run, one logged artifact.  
- `c_hyperparameter_tuning.ipynb` — parent + child runs, an Optuna sweep, the registry.  
- `d_logging_plots.ipynb` — `mlflow.log_figure(...)` across a sweep (per-trial diagnostics + per-dataset EDA).

What changes here is how MLflow and Optuna communicate **during** the sweep instead of only after each trial returns:

- We introduce **Optuna callbacks** — small functions Optuna fires after every trial — and use them for two distinct purposes that upstream conflates under one name.  
- We bring in **XGBoost**, a new model flavor (different `mlflow.<flavor>` integration than the sklearn flavor used in b–d).  
- We use a **conditional hyperparameter space**: which HPs are tunable depends on a previously-suggested categorical (the XGBoost `booster`).  
- We tag the parent run with searchable metadata (`project`, `optimizer_engine`, `model_family`) — the kind of tagging you'd see in a real-world experiment-tracking setup.

This notebook is a modification of the official MLflow tutorial: **[Hyperparameter Tuning with MLflow and Optuna (full notebook)](https://mlflow.org/docs/latest/ml/traditional-ml/tutorials/hyperparameter-tuning/notebooks/hyperparameter-tuning-with-child-runs/#implementing-a-logging-callback)**.  
Where this notebook corrects, supplements, or deliberately diverges from upstream, the relationship is named with an inline bold callout — one of **Bug**, **Stale**, **Missing from**, or **Diverges from upstream tutorial:** — or, in code cells, with a `NOTE (differs from upstream)` comment.  
Sections with no upstream counterpart use a plain heading and no callout.

Three structural differences worth knowing up front:

- **Diverges from upstream tutorial:** upstream calls its single `champion_callback` a *"logging callback"*, but the function only writes to **stdout** — it has no MLflow logging in it at all. In an MLflow context this name is confusing. This notebook keeps `champion_callback` as the **stdout-quietening callback**, and adds a second callback, `mlflow_progress_callback`, that actually logs to MLflow. The two-callback contrast is the central teaching point.  
- **Diverges from upstream tutorial:** upstream uses 500 trials on a synthetic 5000-row "apple demand" dataset. This notebook uses **20 trials** on California housing (continuity with `c_`/`d_`). 20 is enough trials for `champion_callback`'s filtering to be meaningful (5–8 champions, 12–15 non-champions) and for the `mlflow_progress_callback`'s metric history to show a real learning curve, while staying under a minute on a laptop.  
- **Diverges from upstream tutorial:** the final logged model uses XGBoost's native **`ubj`** (universal binary JSON) format. Upstream does the same; called out here because the previous tutorials in this repo used `serialization_format="skops"` for sklearn models, and `ubj` is the XGBoost equivalent — secure, portable, and the recommended XGBoost on-disk format.

## What "logging callback" means

An **Optuna callback** is just a function with the signature `callback(study, frozen_trial)` that Optuna calls after every trial returns. Pass one (or several) via the `callbacks=[...]` argument to `study.optimize(...)`. The callback can do anything Python can — print, send a Slack message, train another model, write to MLflow.

The phrase *"logging callback"* is ambiguous because two very different things both qualify:

| Callback kind | Where the side effect goes | Used for | Survives a session restart? |
|---|---|---|---|
| **Stdout-logging callback** | `sys.stdout` (the notebook output cell, the terminal) | Quietening verbose loops — only print on improvement, not on every trial | No. As soon as the notebook is closed the output is gone unless you saved the `.ipynb`. |
| **MLflow-logging callback** | The MLflow tracking server (run metrics, params, artifacts) | Recording the sweep's progress as part of the run's history — visible in the UI's Charts tab, queryable later via `mlflow.search_runs(...)` | Yes. The metric history lives in the backend store. |

Upstream's `champion_callback` is the first kind — *stdout-logging*, despite the name. This notebook uses it as-is (it's a genuinely useful pattern for long sweeps), and pairs it with `mlflow_progress_callback`, which is the second kind. They're stacked together in the same `callbacks=[...]` list so you can see exactly how the two patterns compose.

**Why two callbacks instead of doing everything inside `objective`?** Because the callback fires **after** the child run's `with`-block has exited — at that point the **parent** run is the active run again. So a `mlflow.log_metric(...)` inside the callback logs to the *parent*, not to any child. That asymmetry is exactly what makes the metric history work: every trial's `best_so_far_mse` ends up on one place (the parent) where the UI can chart them all together.

## Prerequisites

**Diverges from upstream tutorial:**  
Upstream says `pip install xgboost optuna mlflow matplotlib seaborn pandas`. This repo uses **`uv` exclusively** for dependency management — `pip` is explicitly forbidden (see `CLAUDE.md`).  
Everything except `xgboost` was added by previous notebooks; only this one new dep is needed:

```bash
uv add xgboost
```

Commit `pyproject.toml` and `uv.lock` together.

### Start the tracking server

**Missing from upstream tutorial:**  
Before running any cell that calls `mlflow.set_tracking_uri(...)`, start the MLflow server **in a separate terminal**, from `src/`:

```bash
cd src/
mlflow ui --host 127.0.0.1 --port 5001
```

Starting from `src/` keeps per-developer runtime state (`mlflow.db`, `mlartifacts/`) inside the source tree — same convention as the previous notebooks. Update `PORT` below if you use a different port.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.exceptions import RestException

HOST = "127.0.0.1"
PORT = 5001
mlflow.set_tracking_uri(uri=f"http://{HOST}:{PORT}")

# Idempotent experiment creation \u2014 see principle 7 of the
# mlflow-tutorial-improve skill. Re-running this cell is a no-op.
experiment_name = "Logging Callbacks"
try:
    mlflow.create_experiment(name=experiment_name)
except RestException as e:
    if "RESOURCE_ALREADY_EXISTS" not in str(e):
        raise
mlflow.set_experiment(experiment_name)

## Step 1: Load and prepare the data

California housing again — same dataset as `c_` and `d_` so the comparison stays apples-to-apples (no pun on upstream's apple-demand dataset intended).

**What is a `DMatrix`?** XGBoost wraps the (features, labels) pair in an internal data structure called `DMatrix` for performance: it stores values in a column-major, possibly-sparse layout that XGBoost can iterate over more efficiently than a NumPy array. You can pass a DataFrame or NumPy array straight to `xgb.train(...)`, but the DMatrix path is faster and is what upstream uses.

In [ ]:
raw = fetch_california_housing(as_frame=True)
X, y = raw.data, raw.target
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=0)

dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

print(f"train: {X_train.shape},  valid: {X_valid.shape}")

## Step 2: Conditional hyperparameter spaces

XGBoost's `booster` parameter chooses the underlying learner:

| Booster | What it is | Tunable HPs that matter |
|---|---|---|
| `gbtree` | Standard gradient-boosted tree ensemble | `max_depth`, `eta` (learning rate), `gamma`, `grow_policy`, regularization (`lambda`, `alpha`) |
| `dart` | Tree booster with **D**ropout meta-A**R**egularization (drops random trees during training) | Same as `gbtree`, plus dropout-related ones we leave at defaults |
| `gblinear` | **Linear** model fitted with gradient boosting — no trees at all | Only `lambda` and `alpha` (the L2 and L1 regularization); tree-specific HPs *do not apply* |

The interesting consequence is that **`max_depth` is a no-op for `gblinear`** (and so are `eta`, `gamma`, `grow_policy`). If we unconditionally suggested all seven HPs, Optuna would waste exploration budget on parameter dimensions that have zero effect on `gblinear` trials, and the resulting study would be noisier.

Optuna's solution is **dynamic HP spaces**: inside `objective`, suggest the categorical first, *then* condition the rest of the suggestions on its value. TPE (the default sampler) handles this correctly — it tracks each parameter independently and only updates beliefs about a parameter from trials that actually used it.

**The pattern**:

```python
params["booster"] = trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"])
params["lambda"]  = trial.suggest_float("lambda", 1e-8, 1.0, log=True)
params["alpha"]   = trial.suggest_float("alpha",  1e-8, 1.0, log=True)

if params["booster"] in ("gbtree", "dart"):
    params["max_depth"]   = trial.suggest_int("max_depth", 1, 9)
    params["eta"]         = trial.suggest_float("eta", 1e-8, 1.0, log=True)
    # \u2026 etc
```

## Step 3: The first callback — `champion_callback` (stdout heartbeat)

**Purpose: signal-vs-noise reduction in stdout.** In a 500-trial study, the default Optuna logger prints one INFO line per trial. That's 500 lines of mostly-useless output, with the handful of actually-interesting events (a new best) buried inside.

`champion_callback` collapses that to **one print per improvement, plus a percentage delta**. The first trial is also printed (there's no previous "winner" to compare against). Everything else is silently suppressed.

**Pattern: storing state on `study.user_attrs`.** Optuna lets you attach arbitrary key/value state to the study with `study.set_user_attr(...)`. This is how the callback remembers "what was the last value I called the winner" between calls — Python module-level globals would work too but pollute the module namespace and don't survive a study-restart.

**Caveat from upstream (preserved):** *not suitable for distributed computing.* If multiple workers run trials in parallel and call this callback concurrently, the read-check-update on `study.user_attrs["winner"]` is not atomic — two workers can both decide they're champions and both print, with neither's update sticking. For a single-process notebook study like this one, it's fine.

In [ ]:
def champion_callback(study: optuna.Study, frozen_trial: optuna.trial.FrozenTrial) -> None:
    """Print only when a trial improves the running best value. Stdout-only \u2014 does not log to MLflow."""
    winner = study.user_attrs.get("winner", None)

    if study.best_value and winner != study.best_value:
        study.set_user_attr("winner", study.best_value)
        if winner:
            improvement = (abs(winner - study.best_value) / study.best_value) * 100
            print(
                f"Trial {frozen_trial.number} achieved value: {frozen_trial.value:.4f} "
                f"with {improvement:.2f}% improvement over the previous best"
            )
        else:
            print(f"Initial trial {frozen_trial.number} achieved value: {frozen_trial.value:.4f}")

## Step 4: The second callback — `mlflow_progress_callback` (metric history on the parent run)

**Purpose: turn the sweep into a chart in MLflow's UI.** When you log the same metric name multiple times to the same run with different `step` values, MLflow's UI plots them as a **time series** in the Charts tab. We use this to record two parallel series on the parent run:

| Metric on parent | Behaviour over trials | What it teaches you |
|---|---|---|
| `best_so_far_mse` | Monotonically non-increasing (since the study direction is `minimize`) | The classic "learning curve of the sweep" — how fast Optuna is converging. Plateaus mean the search is saturating; sharp drops mean a productive trial. |
| `current_trial_mse` | Noisy, often well above the best-so-far line | The variance of the search. Many bad trials near the best-so-far line means Optuna is exploiting; many wild values means it's still exploring. |

**Why the parent run?** When the callback fires, the child `with`-block has already exited — so the active run is the **parent**. `mlflow.log_metric(...)` lands on the parent automatically. (We could pass an explicit `run_id`, but the fluent API gets it right for free here.)

**Why the `step=` argument?** Without it, MLflow assigns a monotonic timestamp-based step, and the chart's x-axis is wall-clock time. With `step=frozen_trial.number`, the x-axis becomes the trial number — the natural axis for an Optuna sweep, and the one that lets you compare runs of different durations side-by-side.

**Guard against `None` values.** Pruned or failed trials have `frozen_trial.value is None`; logging that to MLflow would raise. The early return keeps the callback robust.

In [ ]:
def mlflow_progress_callback(study: optuna.Study, frozen_trial: optuna.trial.FrozenTrial) -> None:
    """Log best-so-far and current-trial MSE to the parent run with step=trial.number.

    The active run at callback time is the parent (the child's with-block has exited),
    so the fluent log_metric calls below land on the parent automatically.
    """
    if frozen_trial.value is None:
        return  # pruned or failed trial \u2014 nothing to log
    mlflow.log_metric("best_so_far_mse", study.best_value, step=frozen_trial.number)
    mlflow.log_metric("current_trial_mse", frozen_trial.value, step=frozen_trial.number)
    mlflow.log_metric("best_so_far_rmse", math.sqrt(study.best_value), step=frozen_trial.number)

## Step 5: The objective function

Same parent/child pattern as `c_`/`d_` (a `nested=True` child run per trial), with two new pieces:

- The HP space is **conditional** on the booster choice (Step 2).  
- The model is XGBoost, so we use `mlflow.xgboost.log_model(...)` instead of `mlflow.sklearn.log_model(...)`. The `model_format="ubj"` argument tells XGBoost to serialize in its universal binary JSON format — secure (no pickle), portable across versions, and the recommended default since XGBoost 1.6.

We also keep the trial's `run_id` and full metric dict in `trial.user_attrs` so the parent-run cell below can promote them in one pass.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        params: dict = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
            "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
            "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
            "verbosity": 0,
        }
        if params["booster"] in ("gbtree", "dart"):
            params["max_depth"] = trial.suggest_int("max_depth", 1, 9)
            params["eta"] = trial.suggest_float("eta", 1e-8, 1.0, log=True)
            params["gamma"] = trial.suggest_float("gamma", 1e-8, 1.0, log=True)
            params["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])

        booster = xgb.train(params, dtrain, num_boost_round=50)
        preds = booster.predict(dvalid)
        mse = float(mean_squared_error(y_valid, preds))
        rmse = math.sqrt(mse)

        mlflow.log_params(params)
        mlflow.log_metrics({"mse": mse, "rmse": rmse})

        # NOTE (differs from upstream tutorial): we log every child's model, not just
        # the parent's final one. This makes any trial loadable later via its run_id
        # \u2014 useful when the runner-up happens to be the model you actually want.
        mlflow.xgboost.log_model(
            xgb_model=booster,
            name="model",
            model_format="ubj",
        )

        trial.set_user_attr("run_id", child_run.info.run_id)
        trial.set_user_attr("metrics", {"mse": mse, "rmse": rmse})
        return mse

## Step 6: Run the study with both callbacks

Pass both callbacks to `study.optimize(...)` — Optuna fires them **in order** after every trial. `champion_callback` writes to stdout (the cell output below). `mlflow_progress_callback` writes to the parent run's metric history (visible in the MLflow UI's Charts tab).

**Tags on the parent run.** Tags are searchable, free-form key/value pairs. The convention upstream uses — `project`, `optimizer_engine`, `model_family`, `feature_set_version` — is worth copying because it makes the MLflow UI's filter bar usable:

- *"Show me every XGBoost study"* → filter `tags.model_family = 'xgboost'`.  
- *"Show me every Optuna-driven sweep across projects"* → filter `tags.optimizer_engine = 'optuna'`.  

Tags are not the same as params: params are *inputs* to a specific run, tags are *labels* attached for organization. A new feature-set version bumps the tag; a new alpha value bumps the param.

In [ ]:
n_trials = 20

with mlflow.start_run(run_name="study") as parent_run:
    mlflow.set_tags(
        {
            "project": "teach-mlflow",
            "optimizer_engine": "optuna",
            "model_family": "xgboost",
            "feature_set_version": 1,
        }
    )
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="minimize")

    # Silence Optuna's per-trial INFO logs \u2014 our champion_callback replaces them.
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study.optimize(
        objective,
        n_trials=n_trials,
        callbacks=[champion_callback, mlflow_progress_callback],
    )

    # Promote the winning trial's params + metrics onto the parent.
    best_trial = study.best_trial
    mlflow.log_params({f"best_{k}": v for k, v in best_trial.params.items()})
    mlflow.log_metrics({f"best_{k}": v for k, v in best_trial.user_attrs["metrics"].items()})
    mlflow.log_param("best_child_run_id", best_trial.user_attrs["run_id"])

print()
print(f"Best trial: #{best_trial.number}")
print(f"Best MSE:   {study.best_value:.4f}")
print(f"Best RMSE:  {math.sqrt(study.best_value):.4f}")
print(f"Best params: {best_trial.params}")

## Step 7: Refit the best model and log diagnostic plots to the parent

Each trial logged its own model to its own child run (see the `mlflow.xgboost.log_model(...)` call inside `objective`). For the parent we refit one final model on the winning params and attach the **final** plots — feature importance and residuals — to the parent's Artifacts tab. Reasoning:

- *Per-child diagnostic plots* would be ideal but multiply storage; for advanced sweeps you usually keep them only on the best child or the parent.  
- *Per-parent diagnostic plots* are a one-time cost and let anyone reviewing the study see what the winning model actually looks like without having to drill into the children.

**`xgb.plot_importance` and the `importance_type` argument.** For tree boosters (`gbtree`, `dart`) the meaningful importance is `"gain"` — the average improvement in loss each split using the feature produces. For `gblinear` there are no splits, so XGBoost uses `"weight"` (the absolute coefficient value) instead. The conditional below picks the right one.

In [ ]:
best_params = {"objective": "reg:squarederror", "eval_metric": "rmse", "verbosity": 0, **best_trial.params}
best_booster = xgb.train(best_params, dtrain, num_boost_round=50)
best_preds = best_booster.predict(dvalid)
best_residuals = y_valid.values - best_preds

# Feature importance plot \u2014 the metric depends on the booster type.
importance_type = "weight" if best_params["booster"] == "gblinear" else "gain"
fig_imp, ax_imp = plt.subplots(figsize=(8, 5))
xgb.plot_importance(best_booster, importance_type=importance_type, ax=ax_imp, show_values=False)
ax_imp.set_title(f"Feature importance (best trial, importance_type={importance_type!r})")
fig_imp.tight_layout()

# Residuals plot \u2014 same shape as d_logging_plots.ipynb, logged once on the parent.
fig_res, ax_res = plt.subplots(figsize=(7, 5))
ax_res.scatter(best_preds, best_residuals, s=4, alpha=0.3)
ax_res.axhline(0, color="red", linestyle="--", linewidth=1)
ax_res.set_xlabel("predicted")
ax_res.set_ylabel("residual (true \u2212 predicted)")
ax_res.set_title("Best-trial residuals vs predicted")
fig_res.tight_layout()

# Resume the parent run so the figures attach to *it*, not to a new run.
with mlflow.start_run(run_id=parent_run.info.run_id):
    mlflow.log_figure(fig_imp, "diagnostics/feature_importance.png")
    mlflow.log_figure(fig_res, "diagnostics/residuals.png")
    final_model_info = mlflow.xgboost.log_model(
        xgb_model=best_booster,
        name="final_model",
        input_example=X_train.head(),
        model_format="ubj",
    )

print("Parent model URI:", final_model_info.model_uri)

**Resuming an existing run with `mlflow.start_run(run_id=...)`.** The parent's outer `with`-block from Step 6 already exited, so the parent is closed. Reopening it by `run_id` is the canonical pattern for "come back later and add to a run". The run goes from `FINISHED` to `RUNNING` while the block is open and back to `FINISHED` when it exits — entirely safe; the run's start time stays put, and any new log_* calls append to the existing run's metadata.

## Step 8: Inference using the logged final model

Load the final model back via its `model_uri` (the `runs:/...` form points at the run + artifact path; the `models:/...` form would point at the registry if we registered it). Predict on the full feature frame, attach predictions next to the actuals, sanity-check the head.

In [ ]:
loaded = mlflow.xgboost.load_model(final_model_info.model_uri)
inference = loaded.predict(xgb.DMatrix(X))

infer_df = X.copy()
infer_df["actual_demand"] = y.values
infer_df["predicted_demand"] = inference
infer_df.head(3)

## Viewing the run in the UI — what the callbacks bought you

Open <http://127.0.0.1:5001/> and click into **Logging Callbacks** → the `study` parent run.

### The Charts tab is the new view

Switch to the **Charts** tab on the parent. You should see three small-multiples charts:

- **`best_so_far_mse`** — a monotonically non-increasing step curve from trial 0 to trial 19. The steepest drops are the trials `champion_callback` printed about.  
- **`current_trial_mse`** — a noisy line of all 20 trials. Where it touches `best_so_far_mse`, you have a new champion.  
- **`best_so_far_rmse`** — same shape as `best_so_far_mse` but in the target's units ($100k), for human-readable convergence reporting.

Set the **x-axis** to *Step* (not the default *Wall time*) to read the chart in trial-number space. The Step axis is what we logged with `step=frozen_trial.number`; Wall time would just show training-duration variance.

### Stdout vs MLflow side-by-side

Compare what `champion_callback` printed in this notebook with what's on the parent's Charts tab:

- The printed lines vanish when you close the notebook. The MLflow chart persists in `mlflow.db` and is queryable months later.  
- The printed lines name the trial number and improvement %. The chart shows the *full convergence shape* — which prints can't easily convey.  
- Together they cover two different audiences: stdout for the person watching the cell run, MLflow for everyone else (future you, your collaborator, the on-call data scientist next quarter).

### The other tabs

- **Overview** — the four tags (`project`, `optimizer_engine`, `model_family`, `feature_set_version`) appear here and in the experiment's filter bar.  
- **Artifacts** — `diagnostics/feature_importance.png`, `diagnostics/residuals.png`, and `final_model/` (the refit XGBoost saved in UBJ format).  
- **Child runs** (expand-arrow in the runs table) — 20 trials, each with its own model. Click any child to drill down — that trial's own MSE/RMSE and its own `model/` artifact are right there.

## Next steps

- [Optuna callbacks documentation](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html#optuna.study.Study.optimize) — the full `callbacks=[...]` contract, including the pruning callback (`optuna.integration.MLflowCallback` is also worth knowing about — it's a built-in MLflow integration that does some of this automatically, with less customization).  
- [`mlflow.xgboost`](https://mlflow.org/docs/latest/python_api/mlflow.xgboost.html) — the full XGBoost flavor (`save_model`, `load_model`, `autolog`).  
- **Generalizing the metric-history pattern.** Anything you log inside `mlflow_progress_callback` becomes a parent-run time series. Useful variants: log the count of trials with each booster type (one metric per booster, all stepped by trial number) → you get a Charts-tab view of which boosters Optuna is spending its budget on. Same pattern, different metric names.  
- **Pruning unpromising trials.** With XGBoost + Optuna you can implement intermediate-value reporting via `xgb.callback.TrainingCallback` + Optuna's `report` API, then prune trials early with `MedianPruner` or `HyperbandPruner`. That's a follow-on tutorial.